In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d
from render import N_POOLS, render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap
from parameter import P

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED = None
N_CAND = 1500
SIZE = 201
TILE = 256
K = 20

# lowest SH degree. 0 = pure size (absorbed by the volume normalisation),
# 1 = shifts the centroid off the seed. 2 = lowest true shape mode.
L_MIN = 2          
# 4 is the ceiling of the hardcoded Cartesian forms
L     = 4          

# microns per LATERAL voxel
UM_PER_VOX = 0.325                 
# z voxels this many times COARSER than lateral
Z_RATIO    = 1.0                   
# (sz, sy, sx) = voxel SIZE per axis, in lateral units
SPACING    = (Z_RATIO, 1.0, 1.0)   
VOL        = (128, 128, 128)        

# 1) CELL GEOMETRY

RADIUS = P(4.0, 48.0, 0.5, 24.0, "radius", note="radius", 
           comment = "equivalent-sphere radius; volume is normalised to (4/3)pi R^3 whatever the roughness."
           )

ROUGH  = P(0.0, 0.50, 0.01, 0.25, "rough", note="rough",
           comment = "SD of log-radius: 0.25 ~ +-25% radial wobble. Orthogonal to radius and beta."
           )

BETA   = P(0.5, 4.0, 0.1, 1.9, "beta", note="beta",
           comment = "spectral tilt at FIXED total amplitude. Large -> a few fat lobes;\
small -> finer crenulation. Does NOT change how rough the cell is."
           )

ELONG  = P(0.3, 3.0, 0.05, 1.6, "elong", note="elong",
           comment = "volume-preserving aspect ratio. >1 prolate (rod), 1 = sphere, <1 oblate."
           )

# 2) NUCLEUS

NUC_FRAC   = P(0.10, 0.80, 0.01, 0.25, "nuc_frac",   note="nuc frac",
           comment = "nucleus:cell equivalent-RADIUS ratio -> volume ratio is nuc_frac**3."
           )

NUC_ROUGH  = P(0.0, 0.50, 0.01, 0.50, "nuc_rough",  note="nuc rough",
           comment = "nuclear log-radius SD (0.25 ~ +-25% radial wobble.) Orthogonal to radius and beta"
           )

# nuclear spectral tilt. Currently HARDCODED as beta+1.5 -- see caveat below.
NUC_BETA   = P(0.5, 6.0, 0.1, 3.2, "nuc_beta",    note="nuc beta",
           comment = "nuclear spectral tilt. Large -> a few fat lobes;\
small -> finer crenulation"
           )

NUC_CORR   = P(0.0, 1.0, 0.05, 0.40, "nuc_corr",   note="nuc corr",
           comment = "how much the nuclear outline mirrors the cell's"
           )

NUC_OFFSET = P(0.0, 1.0, 0.05, 1., "nuc_offset", note="nuc offset",
           comment = "nuclear displacement as a fraction of the free cytoplasmic room. \
0 puts the nucleus exactly on the tessellation seed -> trivially recoverable."
           )

RIM        = P(0.0, 8.0, 0.5, 0.0, "rim",          note="rim",
           comment = "minimum cytoplasm between nuclear and plasma membrane, in lateral voxels."
           )

# 3) ORIENTATION   (ZYZ: polar+azim aim the long axis, roll spins about it)

POLAR_DEG = P(0.0, 180.0, 5.0, 65.0, "polar_deg", note="polar")
AZIM_DEG  = P(0.0, 360.0, 5.0, 25.0, "azim_deg",  note="azim")
ROLL_DEG  = P(0.0, 360.0, 5.0,  0.0, "roll_deg",  note="roll")


# 4) DRAW THE TAPE

tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, i=0, size=VOL, spacing=SPACING, radius=RADIUS.v, nuc_frac=NUC_FRAC.v, rough=ROUGH.v, elong=ELONG.v, beta=BETA.v,
          polar_deg=POLAR_DEG.v, azim_deg=AZIM_DEG.v, roll_deg=ROLL_DEG.v, rim=RIM.v, nuc_corr=NUC_CORR.v, nuc_offset=NUC_OFFSET.v, beta_nuc = NUC_BETA.v,
          rough_nuc = NUC_ROUGH.v)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v)
fig = plot_surface_xyz_html(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:
marker_config = dict(
    general = dict(
        scope = "general",
        amp=P(.1, 3, .1, 2, "amp"),
        polarity=P(-2, 2, .1, 0.2, "polarity"),
        pol_dir=(
            P(-np.pi, np.pi, .1, 0.0, "pol dir"),
            P(-np.pi, np.pi, .1, 1.0, "pol dir"),
            P(-np.pi, np.pi, .1, 2.0, "pol dir")
        ),
        fluorophores = {
            "r": "APC",
            "g": "FITC",
            "b": "PE",
        }
    ),
    r = [
        dict(
            scope = "blob",
            w=P(0, 1, .05, .80, "weight"),      s=P(0, 2, .05, 0.25, "strength"),
            mu=P(-1, 1.5, .05, 0.50, "mu"),    width=P(.05, 1.5, .05, 1.20, "width"),
            sharp=P(.5, 10, .5, 5.0, "sharp"),  scale=P(.5, 12, .5, 4.5, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .30, "weight"),      s=P(0, 2, .05, 0.05, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),    width=P(.05, 1.5, .05, 0.60, "width"),
            sharp=P(.5, 10, .5, 4.0, "sharp"),  lam=P(2, 25, .5, 5.0, "stripe lam"),
            len=P(3, 40, 1., 12.0, "stripe len"), ang=P(0, 180, 5., 45., "stripe angle"),
            coherence=P(0., 1., 0.1, .60, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .40, "weight"),      s=P(0, 2, .05, 0.10, "strength"),
            mu=P(-1, 1.5, .05, 0.50, "mu"),    width=P(.05, 1.5, .05, 0.50, "width"),
            sharp=P(.5, 10, .5, 3.5, "sharp"),  scale=P(.5, 5, .1, 2.0, "wavelength"),
            coherence=P(0., 1., .1, .40, "sharp")
        )
    ],
    g = [
        dict(
            scope = "blob",
            w=P(0, 1, .05, .50, "weight"),      s=P(0, 2, .05, 0.18, "strength"),
            mu=P(-1, 1.5, .05, 0.10, "mu"),     width=P(.05, 1.5, .05, .50, "width"),
            sharp=P(.5, 10, .5, 7.5, "sharp"),  scale=P(.5, 12, .5, 8.0, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .60, "weight"),      s=P(0, 2, .05, 0.22, "strength"),
            mu=P(-1, 1.5, .05, 0.40, "mu"),     width=P(.05, 1.5, .05, 1.20, "width"),
            sharp=P(.5, 10, .5, 6.0, "sharp"),  lam=P(2, 25, .5, 12.0, "stripe lam"),
            len=P(3, 40, 1., 25.0, "stripe len"), ang=P(0, 180, 5., 90., "stripe angle"),
            coherence=P(0., 1., 0.1, .85, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .90, "weight"),      s=P(0, 2, .05, 0.25, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),     width=P(.05, 1.5, .05, 1.10, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),  scale=P(.5, 5, .1, 3.5, "wavelength"),
            coherence=P(0., 1., .1, .80, "sharp")
        )
    ],
    b = [   
        dict(
            scope = "blob",
            w=P(0, 1, .05, .20, "weight"),      s=P(0, 2, .05, 0.12, "strength"),
            mu=P(-1, 1.5, .05, -0.90, "mu"),    width=P(.05, 1.5, .05, .80, "width"),
            sharp=P(.5, 10, .5, 6.5, "sharp"),  scale=P(.5, 12, .5, 2.0, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .85, "weight"),      s=P(0, 2, .05, 0.15, "strength"),
            mu=P(-1, 1.5, .05, -.90, "mu"),     width=P(.05, 1.5, .05, 1.00, "width"),
            sharp=P(.5, 10, .5, 6.0, "sharp"),  lam=P(2, 25, .5, 1.0, "stripe lam"),
            len=P(3, 40, 1., .35, "stripe len"), ang=P(0, 180, 5., 145., "stripe angle"),
            coherence=P(0., 1., 0.1, .35, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .75, "weight"),      s=P(0, 2, .05, 0.18, "strength"),
            mu=P(-1, 1.5, .05, -.90, "mu"),     width=P(.05, 1.5, .05, .90, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),  scale=P(.5, 5, .1, 4.3, "wavelength"),
            coherence=P(0., 1., .1, .2, "sharp")
        )
    ]
)
img, general = render_image(tape = tape, p_dict = marker_config, cell_mask=cell["cell"], tau = cell["tau"], phi=cell["phi"], d=cell["d"], spacing = SPACING, polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v)

rgb = to_rgb(cell["cell"], img)
print("RGB volume", rgb.shape, f"{rgb.nbytes/1e6:.0f} MB")

In [ ]:
marker_config = dict(
    general = dict(
        scope = "general",
        amp=P(.1, 3, .1, 2, "amp"),
        polarity=P(-2, 2, .1, 0.2, "polarity"),
        pol_dir=(
            P(-np.pi, np.pi, .1, 0.0, "pol dir"),
            P(-np.pi, np.pi, .1, 1.0, "pol dir"),
            P(-np.pi, np.pi, .1, 2.0, "pol dir")
        ),
        fluorophores = {
            "r": "APC",
            "g": "FITC",
            "b": "PE",
        }
    ),
    r = [
        dict(
            scope = "blob",
            w=P(0, 1, .05, .80, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.50, "mu"),    width=P(.05, 1.5, .05, 1.20, "width"),
            sharp=P(.5, 10, .5, 5.0, "sharp"),  scale=P(.5, 12, .5, 4.5, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .30, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),    width=P(.05, 1.5, .05, 0.60, "width"),
            sharp=P(.5, 10, .5, 4.0, "sharp"),  lam=P(2, 25, .5, 5.0, "stripe lam"),
            len=P(3, 40, 1., 12.0, "stripe len"), ang=P(0, 180, 5., 45., "stripe angle"),
            coherence=P(0., 1., 0.1, .60, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .40, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.50, "mu"),    width=P(.05, 1.5, .05, 0.50, "width"),
            sharp=P(.5, 10, .5, 3.5, "sharp"),  scale=P(.5, 5, .1, 2.0, "wavelength"),
            coherence=P(0., 1., .1, .40, "sharp")
        )
    ],
    g = [
        dict(
            scope = "blob",
            w=P(0, 1, .05, .50, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.10, "mu"),     width=P(.05, 1.5, .05, .50, "width"),
            sharp=P(.5, 10, .5, 7.5, "sharp"),  scale=P(.5, 12, .5, 8.0, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .60, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.40, "mu"),     width=P(.05, 1.5, .05, 1.20, "width"),
            sharp=P(.5, 10, .5, 6.0, "sharp"),  lam=P(2, 25, .5, 12.0, "stripe lam"),
            len=P(3, 40, 1., 25.0, "stripe len"), ang=P(0, 180, 5., 90., "stripe angle"),
            coherence=P(0., 1., 0.1, .85, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .90, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, 0.20, "mu"),     width=P(.05, 1.5, .05, 1.10, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),  scale=P(.5, 5, .1, 3.5, "wavelength"),
            coherence=P(0., 1., .1, .80, "sharp")
        )
    ],
    b = [   
        dict(
            scope = "blob",
            w=P(0, 1, .05, .20, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, -0.90, "mu"),    width=P(.05, 1.5, .05, .80, "width"),
            sharp=P(.5, 10, .5, 6.5, "sharp"),  scale=P(.5, 12, .5, 2.0, "blob scale"),
        ),
        dict(
            scope = 'fibre',
            w=P(0, 1, .05, .85, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, -.90, "mu"),     width=P(.05, 1.5, .05, 1.00, "width"),
            sharp=P(.5, 10, .5, 6.0, "sharp"),  lam=P(2, 25, .5, 1.0, "stripe lam"),
            len=P(3, 40, 1., .35, "stripe len"), ang=P(0, 180, 5., 145., "stripe angle"),
            coherence=P(0., 1., 0.1, .35, "coherence")
        ),
        dict(
            scope = 'network',
            w=P(0, 1, .05, .75, "weight"),      s=P(0, 2, .05, 1.9, "strength"),
            mu=P(-1, 1.5, .05, -.90, "mu"),     width=P(.05, 1.5, .05, .90, "width"),
            sharp=P(.5, 10, .5, 7.0, "sharp"),  scale=P(.5, 5, .1, 4.3, "wavelength"),
            coherence=P(0., 1., .1, .2, "sharp")
        )
    ]
)
img, general = render_image(tape = tape, p_dict = marker_config, cell_mask=cell["cell"], tau = cell["tau"], phi=cell["phi"], d=cell["d"], spacing = SPACING, polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v)

rgb = to_rgb(cell["cell"], img)
print("RGB volume", rgb.shape, f"{rgb.nbytes/1e6:.0f} MB")

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = ELONG.v, 
                        polar_deg = POLAR_DEG.v, azim_deg = AZIM_DEG.v, roll_deg = ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
import psfmodels as psfm
import numpy as np
from scipy.signal import fftconvolve
from dataclasses import dataclass

In [ ]:
# Verify
FLUOROPHORES = {
    "DAPI": 0.461, "FITC": 0.519, "PE": 0.578, "APC": 0.660
}

# Optics: 
@dataclass(frozen=True)
class Optics:
    """Known Physical properties of the detector (Macsima) and the experiment."""
    um_per_px: float = 0.325 
    um_per_pz: float = 0.325 
    # um_per_px: float = 1.
    # um_per_pz: float = 1.
    focal_um: float = 0.0
    
    # VERIFY
    na: float = 0.75 #or 0.45 
    wavelength_um: float = 0.530 
    
    # different refractive index of tissue and medium. VERIFY
    n_immersion: float = 1.0
    n_sample: float = 1.33
    
    # Thickness of the section
    section_um: float = 8
    
    
    @property
    def sample_depth_um(self):
        return self.section_um / 2
    
    @property
    def sample_depth_um(self):
        """center of the section"""
        return self.section_um / 2
    
    @property
    def tan_theta(self):
        return float(np.tan(np.arcsin(np.clip(self.na / self.n_immersion, 0.0, 0.999))))

optics = Optics()
N_PLANES = 24

def plane_heights(nz, um_per_pz):
    """Axial coordinate of each slice of a centred volume, in microns."""
    return (np.arange(nz) - (nz - 1) / 2.0) * um_per_pz

def psf_support_px(opt, thickness_um, pad=6):
    """Odd kernel width that contains the most defocused PSF for a section of `thickness_um`."""
    r = 0.5 * thickness_um * opt.tan_theta / opt.um_per_px
    return int(2 * (int(np.ceil(r)) + pad) + 1)

def quad_weights(nz):
    """Quadrature weights for the depth integral -- half weight on the two cut faces."""
    w = np.ones(nz, float)
    w[0] = w[-1] = 0.5
    return w

def psf_kernels(opt, nxy, wavelength_um):
    depth = opt.sample_depth_um + plane_heights(N_PLANES, opt.um_per_px)
    zf = opt.sample_depth_um + opt.focal_um

    assert depth.min() >= 0
    base = dict(nx=int(nxy), dxy=opt.um_per_px, NA=opt.na, wvl=wavelength_um,
                    ni=opt.n_immersion, ni0=opt.n_immersion, ns=opt.n_sample)

    ks = [np.asarray(psfm.make_psf(z=[float(zf)], pz=float(d), model="scalar", **base)[0],
                            np.float32) for d in depth]
    return [k / (k.sum() + 1e-30) for k in ks]

def psf_project(slice_vol, optics, marker = None, general = None):
    nz = slice_vol.shape[0]
    if marker is None or general is None:
        wavelength_um = 0.500
    else:
        wavelength_um = FLUOROPHORES[general["fluorophores"][marker]]
    w = quad_weights(nz)
    nxy = psf_support_px(opt = optics, thickness_um = nz * optics.um_per_pz)
    ks = psf_kernels(opt = optics, nxy = nxy, wavelength_um = wavelength_um)
    out = np.zeros(slice_vol.shape[1:], np.float32)
    for i in range(nz):
        if w[i] == 0:
            continue
        out += np.float32(w[i]) * fftconvolve(slice_vol[i].astype(np.float32), ks[i], mode="same")
    return np.maximum(out, 0.0)

def kryostat(vol, opt, centre_um = 0.0):
    z = plane_heights(vol.shape[0], opt.um_per_pz)
    keep = np.abs(z - centre_um) <= opt.sample_depth_um
    return vol[keep], z[keep]

def NormalizeData(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

subs = {marker: kryostat(v, optics)[0] for marker, v in img.items()}

# img_psf = np.stack([psf_project(vol, optics, marker, general) for marker, vol in subs.items()], -1)
img_psf = np.stack([NormalizeData(psf_project(vol, optics, marker, general)) for marker, vol in subs.items()], -1)

In [ ]:
def mask_collapse(mask, opt=None, mask_pct=0.95):
    """The honest 2D footprint of a 3D object seen through the exact optics.
    """
    cov = psf_project(slice_vol = mask.astype(np.float32), 
                      optics = opt)
    flat = np.sort(cov.ravel())[::-1]
    csum = np.cumsum(flat)
    if csum[-1] <= 0:
        return np.zeros(cov.shape, bool), cov
    k = int(np.searchsorted(csum, mask_pct * csum[-1]))
    thr = flat[min(k, flat.size - 1)]
    return cov >= thr, cov

sub = kryostat(cell["cell"], optics)


plt.imshow(img_psf)
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)